#
Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida




In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
marketing.head(5)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


Se detectaron valores negativos en las columnas "Cantidad" y "monto_total"; se procede a dar tratamiento.

In [ ]:
orders.dtypes

id_pedido              object
id_usuario             object
fecha_hora_pedido      object
pais                   object
dispositivo            object
fuente_referencia      object
nombre_producto        object
categoria_producto     object
cantidad              float64
precio_unitario       float64
monto_descuento       float64
monto_total           float64
dtype: object

In [ ]:
orders.info()
catalog.info()
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
--- 

In [ ]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha']= pd.to_datetime(marketing['fecha'], errors= 'coerce')

Se cambió fecha y fecha_hora_pedido a formato fecha

In [ ]:
catalog.duplicated().sum()

0

In [ ]:
marketing.duplicated().sum()

0

In [ ]:
orders.duplicated().sum()

100

In [ ]:

print(f"Duplicados en id_pedido: {orders['id_pedido'].duplicated().sum()}")


orders = orders.drop_duplicates(subset=['id_pedido'])
print(f"Filas después de eliminar duplicados: {len(orders)}")

Duplicados en id_pedido: 100
Filas después de eliminar duplicados: 25000


No se encontraron duplicados en los dataset "Marteting" y "Catalog"
Se encontraron 100 duplicados en el dataset "Orders" y se eliminaron.

In [ ]:
Q1 = orders[["cantidad", "precio_unitario", "monto_total"]].quantile(0.25)
Q3 = orders[["cantidad", "precio_unitario", "monto_total"]].quantile(0.75)

IQR = Q3-Q1
print("Q1")
print(Q1)
print()

print("Q3")
print(Q3)
print()

print("IQR")
print(IQR)
print()


Q1
cantidad             1.00
precio_unitario    138.47
monto_total        180.67
Name: 0.25, dtype: float64

Q3
cantidad             2.0000
precio_unitario    380.3775
monto_total        518.5800
Name: 0.75, dtype: float64

IQR
cantidad             1.0000
precio_unitario    241.9075
monto_total        337.9100
dtype: float64



In [ ]:
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Límites para detección de outliers:")
print("\nLímite inferior:")
print(limite_inferior)
print("\nLímite superior:")
print(limite_superior)

Límites para detección de outliers:

Límite inferior:
cantidad            -0.50000
precio_unitario   -224.39125
monto_total       -326.19500
dtype: float64

Límite superior:
cantidad              3.50000
precio_unitario     743.23875
monto_total        1025.44500
dtype: float64


In [ ]:
orders = orders[
    (orders["cantidad"] >=Q1["cantidad"]-1.5 *IQR["cantidad"]) &
    (orders["cantidad"] <=Q3["cantidad"]+1.5 *IQR["cantidad"]) &
    (orders["precio_unitario"] >= Q1["precio_unitario"]-1.5*IQR["precio_unitario"])&
    (orders["precio_unitario"] <= Q3["precio_unitario"]+1.5*IQR["precio_unitario"])&
    (orders["monto_total"] >= Q1["monto_total"]-1.5*IQR["monto_total"])&
    (orders["monto_total"] <= Q3["monto_total"]+1.5*IQR["monto_total"])]
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,24936.000000,24936.000000,24936.000000,24936.000000
mean,1.504933,259.362882,4.504331,385.879113
std,0.499986,138.681161,5.224549,255.689113
min,1.000000,20.030000,0.000000,5.240000
25%,1.000000,138.490000,0.000000,180.625000
50%,2.000000,258.775000,0.000000,341.415000
75%,2.000000,380.372500,10.000000,517.640000
max,2.000000,499.960000,15.000000,999.890000


Se trabajaron los outliers usando IQR en el dataset "orders"

In [ ]:
datasets = {
    'marketing': marketing,
    'orders': orders,
    'catalog': catalog
}

for nombre, df in datasets.items():

    print(f"\n======== DATASET: {nombre.upper()} ========\n")

    cat_cols = df.select_dtypes(include=['object', 'category']).columns

    for col in cat_cols:

        print(f"\n--- Columna: {col} ---")

        print("\nValores únicos:")
        print(df[col].nunique())

        print("\nTop categorías:")
        print(df[col].value_counts(dropna=False).head(10))


======== DATASET: MARKETING ========


--- Columna: pais ---

Valores únicos:
3

Top categorías:
Colombia     540
Mexico       540
Argentina    540
Name: pais, dtype: int64

--- Columna: id_campaña ---

Valores únicos:
9

Top categorías:
organic_Colombia         180
social_Colombia          180
organic_Argentina        180
organic_Mexico           180
paid_search_Mexico       180
paid_search_Argentina    180
paid_search_Colombia     180
social_Mexico            180
social_Argentina         180
Name: id_campaña, dtype: int64

--- Columna: canal ---

Valores únicos:
3

Top categorías:
paid_search    507
social         506
organic        506
NaN            101
Name: canal, dtype: int64

======== DATASET: ORDERS ========


--- Columna: id_pedido ---

Valores únicos:
24936

Top categorías:
order_11157    1
order_17729    1
order_10484    1
order_10929    1
order_11042    1
order_13012    1
order_6951     1
order_16127    1
order_15078    1
order_9310     1
Name: id_pedido, dtype: int64

--

se analizó las columnas categoricas de todos los datasets y no se encontraron anomalias

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

Se procede a calcular los costos totales, profit del negocio y el margen.

In [ ]:
Revenue = orders["monto_total"].sum()
print(f"Revenue: ${Revenue:,.2f}")

Revenue: $9,622,281.56


In [ ]:
orders_with_cost = orders.merge(catalog, on='nombre_producto', how='left')

orders_with_cost['costo_total_pedido'] = orders_with_cost['cantidad'] * orders_with_cost['costo_unitario']

# Sumar todos los costos de productos
costo_productos = orders_with_cost['costo_total_pedido'].sum()
print(f"Costo total de productos: ${costo_productos:,.2f}")

gasto_marketing = marketing["gasto"].sum()
print(f"Gasto total marketing: ${gasto_marketing:,.2f}")

Costo total de productos: $3,828,869.01
Gasto total marketing: $2,871,843.53


In [ ]:
costo_total = costo_productos + gasto_marketing
print(f"Costo total: ${costo_total:,.2f}")

Costo total: $6,700,712.54


In [ ]:
profit = Revenue - costo_total
margen_profit = (profit/Revenue) * 100
print(f"Profit: ${profit:,.2f}")
print(f"Margen_profit: {margen_profit:.1f}%")

Profit: $2,921,569.02
Margen_profit: 30.4%


Acorde a los resultados arrojados se puede decir que el negocio es rentable y va por buen camino.

In [ ]:
numero_ordenes = len(orders)
ticket_promedio= Revenue / numero_ordenes

print(f"Numero de ordenes: {numero_ordenes:,.2f}")
print(f"Revenue total: ${Revenue:,.2f}")
print(f"ticket promedio:$ {ticket_promedio:,.2f}")

Numero de ordenes: 24,936.00
Revenue total: $9,622,281.56
ticket promedio:$ 385.88


El valor promedio de una compra es de 385.88

In [ ]:
total_productos = orders["cantidad"].sum()
numero_ordenes = orders["id_pedido"].nunique()
cantidad_promedio = total_productos / numero_ordenes

print(f"Total productos vendidos: {total_productos:,}")
print(f"Numero total de ordenes: {numero_ordenes:,}")
print(f"Cantidad promedio por orden: {cantidad_promedio:.2f}")

Total productos vendidos: 37,527.0
Numero total de ordenes: 24,936
Cantidad promedio por orden: 1.50


La cantidad promedio por orden es 1.50 productos

In [ ]:
productos_por_cantidad = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print(f"Cantidad por producto: {productos_por_cantidad:}")

Cantidad por producto: nombre_producto
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Laptop-Gaming-16GB      4198.0
Tablet-Standard-64GB    4153.0
Phone-Pro-128GB         4140.0
Name: cantidad, dtype: float64


Se envidencia una venta mayor de Vacuum-Pro-Black, Blender-XL-Red, Jacket-Winter-M y Sneakers-Urban-42.

In [ ]:
gasto_por_canal = marketing.groupby("canal")["gasto"].sum().sort_values(ascending= False)
print(f"Gasto en marketing por canal:${gasto_por_canal:}")

Gasto en marketing por canal:$canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


Se evidencia un mayor gasto en los canales "social" y "organic"

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================
query_totals = '''
SELECT nombre_evento,
COUNT(DISTINCT id_usuario) as usuarios_unicos
from  events
where nombre_evento in ('first_visit', 'add_to_cart', 'add_payment_info', 'purchase')
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'add_to_cart' THEN 2
        WHEN 'add_payment_info' THEN 3
        WHEN 'purchase' THEN 4
    END;
'''


totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,add_payment_info,6250
3,purchase,6240


Se observa que los usuarios tienen una alta intencion de compra, en add_to_cart se empiezan a perder lo usuarios pero no es una cantidad grande y los que pasan  la mayoria termina la compra

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first_visit AS (
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento = 'first_visit'
),
cte_add_to_cart AS (
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento = 'add_to_cart'
),
cte_add_payment_info AS (
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento = 'purchase'
)

SELECT
    'first_visit' AS etapa,
    (SELECT COUNT(*) FROM cte_first_visit) AS usuarios_unicos,
    NULL AS conversion_rate

UNION ALL

SELECT
    'add_to_cart' AS etapa,
    (SELECT COUNT(*) FROM cte_add_to_cart) AS usuarios_unicos,
    ROUND(
        (SELECT COUNT(*) FROM cte_add_to_cart) * 100.0 /
        (SELECT COUNT(*) FROM cte_first_visit), 2
    ) AS conversion_rate

UNION ALL

 SELECT
    'add_payment_info' AS etapa,
    (SELECT COUNT(*) FROM cte_add_payment_info) AS usuarios_unicos,
    ROUND(
        (SELECT COUNT(*) FROM cte_add_payment_info) * 100.0 /
        (SELECT COUNT(*) FROM cte_add_to_cart), 2
    ) AS conversion_rate

UNION ALL

SELECT
    'purchase' AS etapa,
    (SELECT COUNT(*) FROM cte_purchase) AS usuarios_unicos,
    ROUND(
        (SELECT COUNT(*) FROM cte_purchase) * 100.0 /
        (SELECT COUNT(*) FROM cte_add_payment_info), 2
    ) AS conversion_rate

'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,etapa,usuarios_unicos,conversion_rate
0,first_visit,7796,NaN
1,add_to_cart,7634,97.92
2,add_payment_info,6250,81.87
3,purchase,6240,99.84


se observa nuevamente una alta intencion de compra, perdida de usurios al pasar a la etapa de add_payment_info ( fuga) y un purchase completo lo cual evidencia una proceso de pago eficiente

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================
query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        CAST(
            DATE_TRUNC(
                'week',
                MIN(CAST(fecha_registro AS TIMESTAMP))
            ) AS DATE
        ) AS cohorte_semana
    FROM users
    GROUP BY id_usuario
),

tamaño_cohorte AS (
    SELECT
        cohorte_semana,
        COUNT(DISTINCT id_usuario) AS total_usuarios
    FROM cohortes
    GROUP BY cohorte_semana
),

retencion AS (
    SELECT
        c.cohorte_semana,
        FLOOR(u.dias_despues_registro / 7) AS semana_retencion,
        u.id_usuario
    FROM user_activity u
    JOIN cohortes c
        ON u.id_usuario = c.id_usuario
    WHERE u.activo = 1
)

SELECT
    r.cohorte_semana,
    r.semana_retencion,
    COUNT(DISTINCT r.id_usuario) AS usuarios_activos,
    t.total_usuarios,

    ROUND(
        COUNT(DISTINCT r.id_usuario) * 100.0
        / t.total_usuarios,
        2
    ) AS retencion_pct

FROM retencion r
JOIN tamanio_cohorte t
    ON r.cohorte_semana = t.cohorte_semana

GROUP BY
    r.cohorte_semana,
    r.semana_retencion,
    t.total_usuarios

ORDER BY 1,2;
'''
# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_semana,semana_retencion,usuarios_activos,total_usuarios,retencion_pct
0,2024-12-30,1.0,99,236,41.95
1,2024-12-30,2.0,91,236,38.56
2,2024-12-30,3.0,95,236,40.25
3,2024-12-30,4.0,97,236,41.10
4,2025-01-06,1.0,145,351,41.31
...,...,...,...,...,...
83,2025-05-19,4.0,165,389,42.42
84,2025-05-26,1.0,142,333,42.64
85,2025-05-26,2.0,134,333,40.24
86,2025-05-26,3.0,142,333,42.64


Las cohortes muestran una retención relativamente estable entre las semanas 1 y 4, manteniéndose alrededor del 40% sin caídas pronunciadas.
Aunque los porcentajes de retención son similares entre 2024 y 2025, durante 2025 se observa un crecimiento importante en el tamaño de las cohortes, indicando una mayor adquisición de usuarios manteniendo niveles consistentes de engagement.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No existe diferencia significativa entre las tasas de conversión de ambas variantes.
   - **H₁ (Hipótesis alternativa):** Sí existe diferencia significativa entre las tasas de conversión.
   
**Test estadístico:** prueba Z
**Nivel de significancia alpha:** 0.05

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [ ]:
df.head(5)

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
conversiones = [
    df[df['variante'] == 'control']['convirtio'].sum(),
    df[df['variante'] == 'tratamiento']['convirtio'].sum()
]

# Total usuarios por grupo
usuarios = [
    df[df['variante'] == 'control']['id_usuario'].count(),
    df[df['variante'] == 'tratamiento']['id_usuario'].count()
]

# Test Z
z_stat, p_value = proportions_ztest(conversiones, usuarios)

print(f"estadistico z: {z_stat}")
print(f"valor p: {p_value}")

estadistico z: -0.8132782986429474
valor p: 0.41605851639119995


No hay evidencia estadísticamente significativa para afirmar que la variante de tratamiento
tenga un impacto diferente en la tasa de conversión respecto al control por lo tanto,
la mejora implementada no muestra un efecto medible en la conversión con este nivel de confianza.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales :**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones :**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría


---